# 05 — Deep Multitask Learning for Indoor Wi-Fi Positioning
## Phase 2: Multitask Architecture, Inductive Transfer & Single-Task Baselines

This notebook demonstrates the implementation, training dynamics, and empirical evaluation of the **Deep Multitask Learning (MTL)** network on the UJIIndoorLoc campus benchmark.

### Core Architectural Concept (Caruana 1997 Hard Parameter Sharing)
- **Shared Representation Trunk:** Dense layers with `LayerNorm`, `GELU` non-linearities, and `Dropout(p=0.2)` learning generalized electromagnetic spatial filters from 459 min-max scaled Wi-Fi access point signals.
- **Building Head:** 3-class classification logits (evaluated with `CrossEntropyLoss`).
- **Floor Head:** 5-class classification logits (evaluated with `CrossEntropyLoss`).
- **Coordinate Head:** 2D continuous metric regression $(x, y)$ in standardized space (evaluated with `MSELoss`).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

from src.data_preprocessing import prepare_data
from src.models.multitask_net import ALL_TASKS, MultitaskNet
from src.train_multitask import load_model, predict, compute_metrics

REPO_ROOT = Path(".") if Path("results").exists() else Path("..")
METRICS_DIR = REPO_ROOT / "results" / "metrics"
MODEL_DIR = REPO_ROOT / "models"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | PyTorch: {torch.__version__}")

/Users/harshita/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/8q/4jf0x5gn2cb5kq8vd9v4ch800000gn/T/matplotlib-7czkomfz because there was an issue with the default path (/Users/harshita/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.
Device: cpu | PyTorch: 2.14.1


## 1. Load Leakage-Free Dataset Partition
Features are scaled to $[0, 1]$ where unobserved access points $(-110\text{ dBm})$ map strictly to $0.0$.
The train/val split is grouped by position ID to prevent identical physical scans from leaking across sets.

In [2]:
data, prep = prepare_data(save=False)
print(f"Active AP features: {len(prep['used_waps'])}")
print(f"Train scans:       {len(data['train']['X']):,}")
print(f"Validation scans:  {len(data['val']['X']):,}")
print(f"Held-out test:     {len(data['test']['X']):,}")

Active AP features: 459
Train scans:       16,432
Validation scans:  2,795
Held-out test:     1,111


## 2. Multitask vs. Single-Task Baselines Comparison
Does joint multitask learning help or hurt? We compare the shared-trunk multitask network against three separate single-task networks (one head each) trained with identical trunks and hyperparameters.

In [3]:
csv_path = METRICS_DIR / "dl_multitask_vs_single_task.csv"
if csv_path.exists():
    df_comp = pd.read_csv(csv_path)
    test_comp = df_comp[df_comp["split"] == "test"][[
        "model", "best_epoch", "building_acc", "floor_acc", "joint_acc",
        "mean_error_m", "median_error_m", "p90_error_m", "evaal_score"
    ]].copy()
    display(test_comp)
else:
    print("dl_multitask_vs_single_task.csv not found.")

             model  best_epoch  ...  p90_error_m  evaal_score
1        multitask         105  ...    17.869964     9.496037
3  single_building           1  ...          NaN     0.000000
5     single_floor          29  ...          NaN     0.356436
7    single_coords          59  ...    17.125506     8.661393

[4 rows x 9 columns]


## 3. Training Dynamics and Loss Curves
We inspect the training and validation history to examine convergence, task competition, and early stopping behavior.

In [4]:
hist_path = METRICS_DIR / "multitask_training_history.csv"
if hist_path.exists():
    history = pd.read_csv(hist_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), dpi=150)
    
    # Loss curves
    ax1.plot(history["epoch"], history["train_total_loss"], label="Train Total Loss", color="#1f77b4")
    ax1.plot(history["epoch"], history["val_total_loss"], label="Val Total Loss", color="#ff7f0e", linestyle="--")
    ax1.set_xlabel("Epoch", fontweight="bold")
    ax1.set_ylabel("Summed Multi-Task Loss", fontweight="bold")
    ax1.set_title("Training vs. Validation Loss", fontweight="bold")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Positioning Error curve
    ax2.plot(history["epoch"], history["val_mean_error_m"], label="Val Mean Error (m)", color="#2ca02c")
    ax2.plot(history["epoch"], history["val_median_error_m"], label="Val Median Error (m)", color="#d62728", linestyle="--")
    ax2.set_xlabel("Epoch", fontweight="bold")
    ax2.set_ylabel("Error (metres)", fontweight="bold")
    ax2.set_title("Validation Positioning Error Dynamics", fontweight="bold")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(REPO_ROOT / 'results' / 'figures' / 'multitask_learning_curves.png')
    plt.close()
    print('Saved loss curves to results/figures/multitask_learning_curves.png')
else:
    print("multitask_training_history.csv not found.")

Saved loss curves to results/figures/multitask_learning_curves.png


## 4. Evaluation of the Saved Best Checkpoint on Test Set
We load `models/multitask_best.pth` and evaluate directly on the 1,111 held-out test fingerprints.

In [5]:
model = load_model("multitask")
pred_test = predict(model, data["test"], prep)
metrics_test = compute_metrics(pred_test, data["test"], prep)

print("=" * 60)
print("HELD-OUT TEST SET EVALUATION (N=1,111)")
print("=" * 60)
print(f"Building Accuracy:          {metrics_test['building_acc']*100:.2f}%")
print(f"Floor Accuracy:             {metrics_test['floor_acc']*100:.2f}%")
print(f"Joint Building-Floor Acc:   {metrics_test['joint_acc']*100:.2f}%")
print(f"Mean Coordinate Error:      {metrics_test['mean_error_m']:.2f} m")
print(f"Median Coordinate Error:    {metrics_test['median_error_m']:.2f} m")
print(f"90th Percentile Error:      {metrics_test['p90_error_m']:.2f} m")
print("=" * 60)

HELD-OUT TEST SET EVALUATION (N=1,111)
Building Accuracy:          99.73%
Floor Accuracy:             91.00%
Joint Building-Floor Acc:   91.00%
Mean Coordinate Error:      8.95 m
Median Coordinate Error:    7.04 m
90th Percentile Error:      17.87 m
